# MiniGPT reads further: follow along

This notebook goes with my post [MiniGPT (Part 7)](https://haddley.github.io/posts/minigpt6/). It counts how fast attention's matches grow, checks that the chunked sliding window gives exactly the right answers, races full attention against the window for memory, and shows the secret-word results: what a long row is for.

**It needs a Mac with Apple Silicon**, because MLX only runs there. Open it in Jupyter from a clone of [minigpt-series](https://github.com/Haddley/minigpt-series), with the packages from its `requirements.txt` installed.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
from notebook_setup import setup
run = setup("part7-sliding-window")
sys.path.insert(0, os.path.abspath("../part5-modern-block"))

working in part7-sliding-window


## 1. Counting the matches

Every working card matches its query against every position up to its own. Each doubling of the row roughly quadruples the matches. With a 256-position window, they grow only in step with the row.

In [2]:
def full_matches(T):
    return T * (T + 1) // 2

def window_matches(T, w=256):
    return sum(min(i + 1, w) for i in range(T))

for T in (4, 8, 1024, 2048, 4096):
    print(f"row {T:>5}: full attention {full_matches(T):>12,} matches, 256-position window {window_matches(T):>10,}")

row     4: full attention           10 matches, 256-position window         10
row     8: full attention           36 matches, 256-position window         36
row  1024: full attention      524,800 matches, 256-position window    229,504
row  2048: full attention    2,098,176 matches, 256-position window    491,648
row  4096: full attention    8,390,656 matches, 256-position window  1,015,936


## 2. The chunked window gives the right answers

With a window as long as the whole row, `chunked_swa` must give exactly full attention's numbers. With a real window, it must match the masked version.

In [3]:
import mlx.core as mx
from model_llama import chunked_swa, sliding_window_mask

mx.random.seed(0)
B, H, T, D = 1, 2, 512, 64
q, k, v = (mx.random.normal((B, H, T, D)) for _ in range(3))
scale = D ** -0.5

full = mx.fast.scaled_dot_product_attention(q, k, v, scale=scale, mask="causal")
print("window = whole row, biggest difference from full attention:",
      mx.abs(chunked_swa(q, k, v, T, scale) - full).max().item())

masked = mx.fast.scaled_dot_product_attention(q, k, v, scale=scale, mask=sliding_window_mask(T, 64, q.dtype))
print("window = 64, biggest difference from the masked version:   ",
      mx.abs(chunked_swa(q, k, v, 64, scale) - masked).max().item())

window = whole row, biggest difference from full attention: 6.556510925292969e-07
window = 64, biggest difference from the masked version:    5.960464477539062e-07


## 3. A short memory race

The same few training steps as `mem_sweep.py`, at three row lengths. The full sweep, up to 16,384 positions, is `python mem_sweep.py`, and needs a 64 GB Mac.

In [4]:
from mem_sweep import run as race
for T in (512, 1024, 2048):
    for mode in ("full", "naive", "chunked"):
        peak, dt = race(T, mode)
        print(f"row {T:>5}  {mode:8}  most memory {peak:6.2f} GB   {dt * 1000:7.1f} ms per step")

row   512  full      most memory   2.47 GB      91.7 ms per step


row   512  naive     most memory   2.47 GB      90.7 ms per step


row   512  chunked   most memory   2.60 GB      99.2 ms per step


row  1024  full      most memory   4.39 GB     208.3 ms per step


row  1024  naive     most memory   4.38 GB     203.4 ms per step


row  1024  chunked   most memory   4.16 GB     187.0 ms per step


row  2048  full      most memory  11.21 GB     556.0 ms per step


row  2048  naive     most memory  11.18 GB     531.8 ms per step


row  2048  chunked   most memory   7.20 GB     371.9 ms per step


## 4. The secret word: what is a long row for?

The post's secret-word test: three machines asked for a word hidden 100 to 970 pieces back. These are the saved results of `run_secret.sh`, which trains the three machines: about 45 minutes, so run it from a terminal with `./run_secret.sh` if you want to repeat it.

In [5]:
import json
names = {"secret256": "256-position row", "secret1024": "1,024 positions, full attention",
         "secretwin256": "1,024 positions, 256-position window"}
gaps = (100, 200, 400, 700, 970)
print(f"{'machine':38}" + "".join(f"{g:>7} back" for g in gaps))
for tag, name in names.items():
    r = json.load(open(f"runs/secret_{tag}.json"))
    print(f"{name:38}" + "".join(f"{r[f'gap_{g}']['right']:>12.1%}" for g in gaps))

machine                                   100 back    200 back    400 back    700 back    970 back
256-position row                            100.0%      100.0%        4.2%        4.2%        4.2%
1,024 positions, full attention             100.0%      100.0%      100.0%      100.0%      100.0%
1,024 positions, 256-position window        100.0%      100.0%        3.8%        3.8%        3.8%
